# Customer Sales Analysis

## Business-focused Exploratory Data Analysis

**Role:** Data Analyst / Business Analytics Portfolio Project  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter

---

## 1. Business Objective

The objective of this project is to analyze customer sales data and identify actionable patterns in **revenue, profitability, customers, products, regions, segments, time and discounting**.

### Key business questions

1. Which categories and regions generate the most revenue?
2. Which categories and regions generate the most profit?
3. Which customer segments contribute most to revenue and profit?
4. Which products are the strongest and weakest performers?
5. How have sales changed over time?
6. Are there seasonal sales patterns?
7. Is there an association between discount levels and profitability?
8. Which areas deserve further investigation or management attention?

> **Analyst principle:** A visualization describes what happened. Business analysis explains why it may matter and what should be investigated next.

## 2. Dataset Overview

The dataset contains transactional sales information such as:

- Category and Sub-Category
- Product and Product ID
- Customer and Customer ID
- Sales, Profit, Quantity and Discount
- Region, Market, Country, State and City
- Segment
- Order and Shipping dates
- Shipping mode and shipping cost
- Order priority

The analysis below is designed to be reproducible from the supplied `superstore.xls` file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

FILE_PATH = "../data/superstore.xls"

df = pd.read_csv(FILE_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

## 3. Data Understanding

Before cleaning or visualization, establish what the data contains.

We inspect:

- Dataset dimensions
- Data types
- Summary statistics
- Unique values
- Missing values
- Duplicate rows

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df.nunique().sort_values(ascending=False)

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

### Interpretation

The dataset should be understood before any business conclusion is made.

Identifier columns such as `Order_Id`, `Customer_Id` and `Row_Id` help identify entities, while measures such as `Sales`, `Profit`, `Quantity`, `Discount` and `Shipping_Cost` can be aggregated or compared.

Categorical fields such as `Category`, `Region` and `Segment` are generally analyzed using grouped comparisons rather than numerical correlation.

## 4. Data Cleaning

Cleaning decisions should be based on evidence, not assumptions.

We will:

1. Check missing values.
2. Check duplicate records.
3. Convert date fields.
4. Investigate duplicate/redundant date columns.
5. Remove fields that contain no analytical variation when appropriate.
6. Create useful time and operational features.

In [ ]:
# Convert available date columns safely

for col in ["Order_Date", "Order_date", "Shipping_Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df[["Order_Date", "Shipping_Date"]].head()

In [ ]:
# Investigate whether Order_Date and Order_date are duplicates.

if "Order_Date" in df.columns and "Order_date" in df.columns:
    print(
        "Order_Date and Order_date identical:",
        df["Order_Date"].equals(df["Order_date"])
    )

In [ ]:
# Remove redundant duplicate date column only if it is demonstrably identical.

if (
    "Order_Date" in df.columns
    and "Order_date" in df.columns
    and df["Order_Date"].equals(df["Order_date"])
):
    df.drop(columns=["Order_date"], inplace=True)

# Remove Record_Count only when it has no variation.
if "Record_Count" in df.columns and df["Record_Count"].nunique() == 1:
    df.drop(columns=["Record_Count"], inplace=True)

print(df.columns.tolist())

In [ ]:
# Create time and shipping features

df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()

df["Shipping_Days"] = (
    df["Shipping_Date"] - df["Order_Date"]
).dt.days

df.head()

### Cleaning conclusion

The objective of cleaning is not to make the dataset smaller. It is to make the dataset **consistent and analytically useful** while preserving information that may matter to the business.

For every removed column, a professional analyst should be able to explain **why it was removed**.

## 5. Business KPIs

Before detailed analysis, establish a high-level view of the business.

### Core KPIs

- Total Sales
- Total Profit
- Profit Margin
- Total Quantity
- Unique Orders
- Unique Customers

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_quantity = df["Quantity"].sum()

unique_orders = df["Order_Id"].nunique()
unique_customers = df["Customer_Id"].nunique()

profit_margin = total_profit / total_sales * 100

kpis = pd.DataFrame({
    "KPI": [
        "Total Sales",
        "Total Profit",
        "Profit Margin",
        "Total Quantity",
        "Unique Orders",
        "Unique Customers"
    ],
    "Value": [
        total_sales,
        total_profit,
        profit_margin,
        total_quantity,
        unique_orders,
        unique_customers
    ]
})

kpis

### KPI interpretation

These KPIs provide the baseline for the rest of the project.

A key principle is:

> **Revenue tells us how much the company sold; profit tells us what remained after the costs represented in the dataset.**

Therefore, sales performance should always be considered alongside profitability.

# 6. Revenue Analysis

## 6.1 Sales by Category

### Business question

> Which product category generates the most revenue?

In [ ]:
category_sales = (
    df.groupby("Category")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

category_sales

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x=category_sales.index, y=category_sales.values)

plt.title("Total Sales by Category")
plt.xlabel("Category")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

### How to interpret

**Observation:** Identify the highest and lowest categories.

**Evidence:** Compare their sales totals.

**Business meaning:** A high-sales category represents an important revenue stream.

**Next investigation:** Check whether it also has strong profit and profit margin.

**Do not conclude:** "The highest-sales category is automatically the best category."

## 6.2 Sales by Region

### Business question

> Which geographical regions generate the most revenue?

In [ ]:
region_sales = (
    df.groupby("Region")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

region_sales

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(x=region_sales.index, y=region_sales.values)

plt.title("Total Sales by Region")
plt.xlabel("Region")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Business interpretation framework

If one region leads:

- Determine how much revenue it contributes.
- Compare its order volume.
- Examine its product/category mix.
- Compare its profit.
- Investigate whether successful practices could be replicated elsewhere.

This is **drill-down analysis**: moving from a high-level finding to the factors behind it.

## 6.3 Sales by Customer Segment

### Business question

> Which customer segment generates the most revenue?

In [ ]:
segment_sales = (
    df.groupby("Segment")["Sales"]
      .sum()
      .sort_values(ascending=False)
)

segment_sales

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=segment_sales.index, y=segment_sales.values)

plt.title("Total Sales by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

### Business interpretation

The largest sales segment is an important revenue source, but revenue alone does not tell us:

- how profitable the segment is,
- how many customers it contains,
- how many orders it generates,
- or how large its average order is.

Those questions should be answered before making strategic recommendations.

## 6.4 Sales by Year

### Business question

> Is revenue growing or declining over time?

In [ ]:
yearly_sales = (
    df.groupby("Year")["Sales"]
      .sum()
      .sort_index()
)

yearly_sales

In [ ]:
plt.figure(figsize=(9, 5))
sns.lineplot(x=yearly_sales.index, y=yearly_sales.values, marker="o")

plt.title("Total Sales by Year")
plt.xlabel("Year")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.show()

In [ ]:
yearly_growth = yearly_sales.pct_change() * 100

yearly_growth_table = pd.DataFrame({
    "Sales": yearly_sales,
    "YoY_Growth_%": yearly_growth
})

yearly_growth_table

### Business interpretation

A rising trend indicates revenue growth across the observed years.

The percentage-growth table adds another layer: it shows **how quickly revenue changed**, not just whether it increased.

Before claiming that growth will continue in the future, remember that historical growth does not guarantee future growth.

## 6.5 Monthly Sales Pattern

### Business question

> Are there recurring periods of stronger or weaker demand?

In [ ]:
monthly_sales = (
    df.groupby("Month")["Sales"]
      .sum()
      .sort_index()
)

monthly_sales

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(x=monthly_sales.index, y=monthly_sales.values, marker="o")

plt.title("Monthly Sales Pattern")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.xticks(range(1, 13))
plt.tight_layout()
plt.show()

### Analyst caution

This combines all years.

Therefore, if December is highest, the safe statement is:

> "December has the highest total sales across the full dataset."

It is not enough evidence to claim that December is the best month **every year**.

To test consistency, compare Year × Month.

In [ ]:
year_month_sales = (
    df.groupby(["Year", "Month"])["Sales"]
      .sum()
      .reset_index()
)

year_month_sales.head(15)

## 6.6 Top 10 Products by Sales

### Business question

> Which products contribute the most revenue?

In [ ]:
top_products_sales = (
    df.groupby("Product_Name")["Sales"]
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

top_products_sales

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(x=top_products_sales.values, y=top_products_sales.index)

plt.title("Top 10 Products by Sales")
plt.xlabel("Total Sales")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

### Business interpretation

Top-selling products may deserve:

- inventory monitoring,
- availability checks,
- marketing attention,
- pricing analysis.

But a top seller is not necessarily the most profitable product. We will test that later.

# 7. Profitability Analysis

## 7.1 Profit by Category

### Business question

> Which category contributes the most profit?

In [ ]:
category_profit = (
    df.groupby("Category")["Profit"]
      .sum()
      .sort_values(ascending=False)
)

category_profit

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x=category_profit.index, y=category_profit.values)

plt.title("Total Profit by Category")
plt.xlabel("Category")
plt.ylabel("Total Profit")
plt.tight_layout()
plt.show()

### Analyst lesson

Now compare this with **Sales by Category**.

A business should care about both:

**Revenue performance** and **profitability**.

A category with high sales but poor profit deserves investigation rather than automatic expansion.

## 7.2 Profit by Region

### Business question

> Which regions are actually profitable?

In [ ]:
region_profit = (
    df.groupby("Region")["Profit"]
      .sum()
      .sort_values(ascending=False)
)

region_profit

In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(x=region_profit.index, y=region_profit.values)

plt.title("Total Profit by Region")
plt.xlabel("Region")
plt.ylabel("Total Profit")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Business interpretation

Compare the ranking of regions by:

- Sales
- Profit
- Profit margin

If the rankings differ, investigate:

- product mix,
- discounting,
- shipping costs,
- customer mix,
- pricing.

## 7.3 Category Performance — Sales, Profit and Margin

This combines several metrics into one management-friendly table.

In [ ]:
category_performance = (
    df.groupby("Category")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Quantity=("Quantity", "sum")
      )
)

category_performance["Profit_Margin_%"] = (
    category_performance["Profit"] /
    category_performance["Sales"] * 100
)

category_performance.sort_values("Sales", ascending=False)

### Why this is valuable

Now we can distinguish:

- high-sales categories,
- high-profit categories,
- high-margin categories.

These are not always the same thing.

## 7.4 Product Profitability

### Business question

> Which products create profit and which products create losses?

In [ ]:
product_performance = (
    df.groupby("Product_Name")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Quantity=("Quantity", "sum")
      )
)

product_performance["Profit_Margin_%"] = (
    product_performance["Profit"] /
    product_performance["Sales"] * 100
)

product_performance.sort_values("Profit", ascending=False).head(10)

In [ ]:
loss_making_products = (
    product_performance[product_performance["Profit"] < 0]
    .sort_values("Profit")
)

loss_making_products.head(10)

In [ ]:
plt.figure(figsize=(11, 7))

worst_products = loss_making_products.head(10)

sns.barplot(
    x=worst_products["Profit"].values,
    y=worst_products.index
)

plt.title("10 Products with Largest Total Losses")
plt.xlabel("Total Profit")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

### Business interpretation

Loss-making products should be investigated, not automatically removed.

Possible questions:

- Are discounts unusually high?
- Are shipping costs high?
- Is the product sold heavily in a particular region?
- Is the product strategically important?
- Is the loss caused by a small number of transactions?

This is where an analyst moves from **identifying a problem to diagnosing it**.

# 8. Discount and Profitability Analysis

## Business question

> Does discount level appear to be associated with profitability?

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Discount",
    y="Profit",
    alpha=0.3
)

plt.title("Discount vs Profit")
plt.xlabel("Discount")
plt.ylabel("Profit")
plt.tight_layout()
plt.show()

### First observation

Look for:

- the overall direction,
- clusters,
- extreme profits/losses,
- whether high discount levels contain more negative-profit observations.

A scatter plot can suggest a relationship, but it does not establish causation.

In [ ]:
discount_profit = (
    df.groupby("Discount")["Profit"]
      .agg(["mean", "median", "count"])
      .sort_index()
)

discount_profit

In [ ]:
df["Is_Loss"] = df["Profit"] < 0

loss_rate_by_discount = (
    df.groupby("Discount")["Is_Loss"]
      .mean()
      .mul(100)
      .sort_index()
)

loss_rate_by_discount

In [ ]:
plt.figure(figsize=(10, 5))

sns.lineplot(
    x=loss_rate_by_discount.index,
    y=loss_rate_by_discount.values,
    marker="o"
)

plt.title("Loss-Making Transactions by Discount Level")
plt.xlabel("Discount")
plt.ylabel("Loss-Making Transactions (%)")
plt.tight_layout()
plt.show()

### Business interpretation

If loss rates rise as discounts increase, an appropriate conclusion is:

> **Higher discount levels are associated with a greater proportion of loss-making transactions.**

Do **not** write:

> "Discount causes losses."

There may be confounding factors such as product, region, quantity, shipping cost or market.

A management recommendation should therefore be to **evaluate the profitability of high-discount transactions**, not blindly eliminate discounts.

# 9. Correlation Analysis

## Why correlation?

With a large dataset, manually testing every numerical pair is inefficient.

A correlation matrix provides an automated **screening mechanism** for numerical variables.

In [ ]:
numeric_columns = [
    c for c in [
        "Discount",
        "Profit",
        "Quantity",
        "Sales",
        "Shipping_Cost",
        "Shipping_Days"
    ]
    if c in df.columns
]

corr = df[numeric_columns].corr()

corr

In [ ]:
plt.figure(figsize=(9, 7))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Matrix — Key Numerical Variables")
plt.tight_layout()
plt.show()

In [ ]:
# Automatically rank unique correlation pairs

corr_pairs = (
    corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )
    .stack()
    .sort_values(key=lambda x: x.abs(), ascending=False)
)

corr_pairs

### How an analyst uses correlation

**Step 1 — Screen automatically**

Use the correlation matrix to find potentially interesting numerical relationships.

**Step 2 — Investigate**

Create a targeted visualization or grouped analysis.

**Step 3 — Add context**

Ask whether the relationship makes business sense.

**Step 4 — Validate**

Use additional statistical analysis if the business decision requires stronger evidence.

### Important

Correlation measures association, not causation.

A correlation close to zero also does not prove that no relationship exists; it mainly indicates little **linear** association.

# 10. Customer and Segment Analysis

## Segment performance

Compare sales, profit, orders, customers and margin together.

In [ ]:
segment_performance = (
    df.groupby("Segment")
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Orders=("Order_Id", "nunique"),
          Customers=("Customer_Id", "nunique")
      )
)

segment_performance["Profit_Margin_%"] = (
    segment_performance["Profit"] /
    segment_performance["Sales"] * 100
)

segment_performance.sort_values("Sales", ascending=False)

### Average Order Value

Average Order Value (AOV) helps distinguish a segment with many small orders from one with fewer but larger orders.

In [ ]:
aov = (
    df.groupby("Segment")
      .agg(
          Sales=("Sales", "sum"),
          Orders=("Order_Id", "nunique")
      )
)

aov["Average_Order_Value"] = aov["Sales"] / aov["Orders"]

aov.sort_values("Average_Order_Value", ascending=False)

## Customer-level performance

High-value customers can be important for retention and account management.

In [ ]:
customer_performance = (
    df.groupby(["Customer_Id", "Customer_Name"])
      .agg(
          Sales=("Sales", "sum"),
          Profit=("Profit", "sum"),
          Orders=("Order_Id", "nunique")
      )
      .sort_values("Sales", ascending=False)
)

customer_performance.head(10)

### Analyst caution

High revenue from a customer does not automatically mean high profitability.

Customer-level decisions should consider both sales and profit.

# 11. Operational Analysis

## Shipping mode

The dataset contains shipping mode, shipping cost and shipping dates.

We can investigate whether operational patterns deserve attention.

In [ ]:
shipping_summary = (
    df.groupby("Shipping_Mode")
      .agg(
          Orders=("Order_Id", "nunique"),
          Avg_Shipping_Days=("Shipping_Days", "mean"),
          Avg_Shipping_Cost=("Shipping_Cost", "mean"),
          Total_Shipping_Cost=("Shipping_Cost", "sum")
      )
      .sort_values("Orders", ascending=False)
)

shipping_summary

In [ ]:
plt.figure(figsize=(9, 5))

sns.barplot(
    x=shipping_summary.index,
    y=shipping_summary["Orders"]
)

plt.title("Orders by Shipping Mode")
plt.xlabel("Shipping Mode")
plt.ylabel("Number of Orders")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Business interpretation

The most frequently used shipping mode is not automatically the best-performing mode.

Compare:

- order volume,
- delivery time,
- shipping cost,
- customer segment,
- region,
- profitability.

This illustrates an important principle:

> **Frequency is not the same as performance.**

# 12. Key Business Insights

The following section should contain only findings supported by the analysis above.

## How to write an insight

Use:

**Observation → Evidence → Interpretation → Business Impact → Recommendation**

### Example

**Observation:** Technology leads category sales.

**Evidence:** Its total sales are higher than the other categories.

**Interpretation:** Technology is a major revenue driver.

**Business impact:** Its product availability and profitability are important to overall performance.

**Recommendation:** Investigate the products, margins and discount levels driving Technology's performance before making inventory or investment decisions.

## Dataset-specific findings to verify from your executed outputs

### 1. Technology performance
Technology is the leading category by both sales and profit in this dataset. This makes it an important revenue and profit driver.

### 2. Regional performance
Central is the leading region by both sales and profit. Its product mix, customer mix and commercial practices are worth investigating.

### 3. Customer segment
Consumer is the largest revenue-generating segment. Further analysis should compare its order volume, customer count, average order value and profitability with other segments.

### 4. Revenue growth
Sales increase across the observed years from 2011 through 2014. Year-over-year growth should be considered alongside profit growth to assess the quality of that expansion.

### 5. Discount risk
Higher discount levels are associated with a greater proportion of loss-making transactions. This is a signal to evaluate discount effectiveness and margin protection.

> **Important:** These are findings from the supplied dataset. Before publishing the portfolio version, run every cell and confirm the values shown in your final notebook.

# 13. Business Recommendations

Based on the findings, management could consider:

### Recommendation 1 — Protect high-performing categories
Investigate the factors behind Technology's strong sales and profit performance, including its top products, regional demand and discounting.

### Recommendation 2 — Learn from high-performing regions
Analyze Central's product mix, customer mix and discount strategy to determine whether successful practices can be replicated in lower-performing regions.

### Recommendation 3 — Review high-discount transactions
Evaluate whether additional sales generated by high discounts justify the associated reduction in profitability.

### Recommendation 4 — Manage products using profit as well as sales
Monitor high-revenue products using profit and profit margin. Investigate products that generate substantial sales but weak or negative profit.

### Recommendation 5 — Plan around demand patterns
Use year-by-year and month-by-month analysis to support inventory and marketing planning, but confirm that seasonal patterns are consistent across years before making major decisions.

# 14. Limitations

A professional analysis should clearly state what the dataset cannot prove.

### Key limitations

- Historical data does not guarantee future performance.
- Correlation does not establish causation.
- Sales alone does not measure business success.
- The dataset does not necessarily include every cost relevant to true net profitability.
- Seasonal patterns should be validated across individual years.
- Loss-making products may have strategic value that is not visible in this dataset.
- Recommendations should be combined with business context before implementation.

# 15. Final Executive Conclusion

The analysis indicates strong revenue growth across the observed period, with Technology emerging as the strongest category in both sales and profit. Central is the leading region by sales and profit, while Consumer represents the largest revenue-generating customer segment.

The analysis also identifies a relationship between discounting and profitability: higher discount levels are associated with a greater proportion of loss-making transactions. This suggests that discount strategy deserves closer profitability analysis.

Overall, the company should protect and understand the drivers of high-performing categories and regions while monitoring product-level margins and the profitability of high-discount transactions.

The next stage of analysis would be to drill down into **product × region × category × discount** combinations to identify the specific business conditions behind the strongest and weakest outcomes.

# Portfolio Quality Checklist

Before publishing this notebook:

- [ ] All cells execute from top to bottom without errors.
- [ ] No temporary/debug cells remain.
- [ ] No tutorial instructions remain in the final notebook.
- [ ] Every important chart has a clear title and axis labels.
- [ ] Conclusions are based on actual output.
- [ ] No causal claims are made from correlation alone.
- [ ] Sales and profit are analyzed separately.
- [ ] Recommendations are evidence-based.
- [ ] File paths work for another user.
- [ ] README explains the project and how to run it.
- [ ] Dataset licensing/usage terms are checked before public upload.